# Getting the Minergie attribute of the flatfox listings via the flatfox Web API

**This Jupyter notebook does the following:**

- Reads the flatfox listings from `Data/flatfox_switzerland_rent_apartments_pre1991.csv`
- Extracts the listing ID from each detail URL
- Queries each listing from the public flatfox Web API (`https://flatfox.ch/api/v1/public-listing/<id>/`), with several attempts if the server is busy and a pause between requests
- Saves the listing attributes (e.g. `minergie`, `lift`, `garage`) and the net rent to `Data/flatfox_api_attributes.csv`

Run this notebook **before** `flatfox_pre1991_data_preparation.ipynb`, which uses the saved file to create the column `minergie`. The saved file is part of the repository, so the notebook only needs to be run again to update the data (listings that were removed from flatfox return status 404).

The structure follows `Week_07/getting_supermarket_locations_WebAPI.ipynb`.

## Libraries and settings

In [ ]:
# Libraries
import os
import time
import requests
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

## Importing the flatfox listings and extracting the listing ID

In [ ]:
# Read the data to a pandas data frame
df = pd.read_csv('./Data/flatfox_switzerland_rent_apartments_pre1991.csv', sep=',', encoding='utf-8')

# The listing ID is the last number in the detail URL,
# e.g. 'https://flatfox.ch/en/flat/6614-brissago/1848132/' -> 1848132
df['pk'] = df['detail_url-href'].str.extract(r'/(\d+)/?$', expand=False).astype('Int64')

# Unique listing IDs
pks = df['pk'].dropna().unique()
print('Listings:', df.shape[0], '| unique listing IDs:', len(pks))

## Query each listing from the flatfox Web API

- One request per listing, with a pause of 1 second between requests so that the server is not overloaded.
- Status 404 means that the listing is no longer online; its attributes stay empty.
- The query of about 400 listings takes about 7 minutes.

In [ ]:
# flatfox Web API URL of one listing
url = 'https://flatfox.ch/api/v1/public-listing/{}/'

max_attempts = 3
results = []

for n, pk in enumerate(pks, start=1):
    row = {'pk': pk, 'http_status': None, 'listing_status': None,
           'attributes': None, 'rent_net': None}

    # Web API request (several attempts, as the server may be busy)
    for attempt in range(1, max_attempts + 1):
        try:
            r = requests.get(url.format(pk), timeout=30)
            row['http_status'] = r.status_code
            if r.status_code == 404:
                break
            r.raise_for_status()
            data = r.json()
            row['listing_status'] = data.get('status')
            row['attributes'] = ','.join(a['name'] for a in data.get('attributes', []))
            row['rent_net'] = data.get('rent_net')
            break
        except (requests.exceptions.RequestException, ValueError, KeyError) as e:
            print(f'Listing {pk}: attempt {attempt} of {max_attempts} failed: {e}')
            if attempt < max_attempts:
                time.sleep(10 * attempt)

    results.append(row)

    # Show progress
    if n % 50 == 0:
        print(f'{n} of {len(pks)} listings queried')

    # Pause between requests
    time.sleep(1)

# Store results in data frame
df_api = pd.DataFrame(results)
df_api['pk'] = df_api['pk'].astype('Int64')
df_api['http_status'] = df_api['http_status'].astype('Int64')
df_api['rent_net'] = pd.to_numeric(df_api['rent_net'], errors='coerce').astype('Int64')

print(df_api.shape)
df_api.head()

## Check the results

In [ ]:
# HTTP status (200 = found, 404 = listing no longer online)
print(df_api['http_status'].value_counts(dropna=False), '\n')

# Listings with the attribute 'minergie'
df_api['minergie_attribute'] = df_api['attributes'].fillna('').str.split(',').apply(lambda x: 'minergie' in x)
print('Listings with attribute minergie:', df_api['minergie_attribute'].sum(), '\n')

# Frequency of all attributes
df_api['attributes'].dropna().str.split(',').explode().value_counts()

### Save data to file

In [ ]:
df_api.to_csv('./Data/flatfox_api_attributes.csv',
              sep=",",
              encoding='utf-8',
              index=False)

print('Saved:', df_api.shape[0], 'listings')

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')